# Ch.10 — Transformers & Attention
**Track:** ML from Scratch · California Housing dataset (8 features treated as a sequence of tokens)

## Core Idea

The LSTM (Ch.6) solved the vanishing gradient problem but still processes tokens **sequentially** — step 1 must complete before step 2. Transformers discard recurrence entirely and replace it with **attention**: a single parallel operation that lets every position directly compare itself to every other position at once.

```
RNN:         x₁ → x₂ → x₃ → ... → xT     (sequential bottleneck)
Transformer: [x₁, x₂, x₃, ..., xT]        (all positions in parallel)
                      ↓
             Multi-Head Attention: every position attends to every other
```

**The key equations:**

Scaled dot-product attention:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Positional encoding (sinusoidal):

$$\text{PE}_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad \text{PE}_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

## Running Example

The real estate platform has 8 tabular features per district in the California Housing dataset. We treat each feature as a **token** — a sequence of length 8. This is architecturally unconventional but pedagogically perfect: no new dataset, and the resulting attention heatmap has an immediately interpretable meaning ("which feature is attending to which other feature?").

Feature tokens: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Load data ─────────────────────────────────────────────────────────────────
data = fetch_california_housing()
X_raw, y = data.data, data.target      # (20640, 8), (20640,)
feature_names = list(data.feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw) # scale before projecting to tokens

# Treat each of the 8 features as a token with 1 raw dimension
# Shape: (N, T=8, 1)
X_tokens = X_scaled[:, :, np.newaxis].astype(np.float32)

print(f"Samples: {X_tokens.shape[0]}")
print(f"Sequence length T (tokens): {X_tokens.shape[1]}")
print(f"Raw token dimension: {X_tokens.shape[2]}")
print(f"Feature names (tokens): {feature_names}")

## The Math — Part 1: Positional Encoding

Attention is **permutation-equivariant**: shuffle the tokens and the output shuffles identically. Without positional information, the model cannot distinguish "feature 0 is MedInc" from "feature 5 is AveOccup". We inject position by **adding** a fixed encoding matrix to the token embeddings.

In [ ]:
def positional_encoding(T, d_model):
    """Sinusoidal positional encoding — shape (T, d_model)."""
    PE = np.zeros((T, d_model))
    for pos in range(T):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            PE[pos, i]   = np.sin(pos / denom)
            PE[pos, i+1] = np.cos(pos / denom)
    return PE

T, D_MODEL = 8, 16
PE = positional_encoding(T, D_MODEL)

# ── Plot: encoding matrix heatmap ─────────────────────────────────────────────
plt.figure(figsize=(11, 3.5))
sns.heatmap(PE, cmap='RdBu_r', center=0, annot=False,
            xticklabels=[f'd{i}' for i in range(D_MODEL)],
            yticklabels=feature_names)
plt.title('Positional Encoding — 8 feature tokens × 16 dimensions\n'
          'Low dims: slow oscillation (coarse position)  |  High dims: fast oscillation (fine position)')
plt.xlabel('Encoding dimension'); plt.ylabel('Token position (feature)')
plt.tight_layout()
plt.show()

print("PE shape:", PE.shape)
print(f"\nRow 0 (MedInc at pos 0): sin(0)=0, cos(0)=1 for all dims")
print(f"  PE[0, :4] = {PE[0, :4].round(3)}")
print(f"  PE[1, :4] = {PE[1, :4].round(3)}  ← already different at pos 1")

## The Math — Part 2: Scaled Dot-Product Attention

For each token, we project the input into three roles — **Query** (what am I looking for?), **Key** (what do I offer?), **Value** (what do I contribute if selected?). The attention weights are dot-product similarities between queries and keys, scaled by $\sqrt{d_k}$ to prevent softmax saturation.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K: (T, d_k)
    V:    (T, d_v)
    Returns: output (T, d_v), weights (T, T)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (T, T) raw similarity

    if mask is not None:
        scores = scores + mask                 # -inf where masked → 0 after softmax

    # Numerically stable softmax
    scores = scores - scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores)
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)  # (T, T)

    output = weights @ V                       # (T, d_v)
    return output, weights

# Demo: random QKV projections on a single sample with PE added
rng = np.random.default_rng(42)
d_k = 8

# Project raw feature value (1-dim) + PE (16-dim) → use PE alone for demo clarity
x_demo = PE.copy()          # (8, 16) — PE already encodes position + we add feature signal below
x_demo[:, 0:1] += X_tokens[0]   # inject the first sample's feature values into dim 0

WQ = rng.normal(0, 0.1, (D_MODEL, d_k))
WK = rng.normal(0, 0.1, (D_MODEL, d_k))
WV = rng.normal(0, 0.1, (D_MODEL, d_k))

Q = x_demo @ WQ    # (8, d_k)
K = x_demo @ WK
V = x_demo @ WV

output_enc, weights_enc = scaled_dot_product_attention(Q, K, V, mask=None)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("Attention output shape:", output_enc.shape)
print("Attention weights shape:", weights_enc.shape)
print(f"\nWeights row 0 (MedInc attending to all features): {weights_enc[0].round(3)}")
print(f"Sum of weights row 0 = {weights_enc[0].sum():.4f}  ← always 1.0")

## The Killer Visual — Attention Heatmap

Each row is a **query token** (the feature asking "who should I attend to?"). Each column is a **key token** (the feature being attended to). High weight = the model learned a strong relationship between those two features.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(weights_enc, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names,
            vmin=0, vmax=weights_enc.max())
plt.title('Attention Weights — which feature "attends to" which?\n'
          'Row = Query (I am this feature)  |  Col = Key (I am attending to this)')
plt.xlabel('Key (attended-to feature)'); plt.ylabel('Query (attending feature)')
plt.tight_layout()
plt.show()

## Encoder vs Decoder — One Mask Difference

| | Encoder | Decoder |
|---|---|---|
| Mask | None — every position sees every other | Causal mask — position t sees only ≤ t |
| Trained for | Representation, classification, embeddings | Token generation, LLMs |
| Examples | BERT, embedding models | GPT-4, Llama, Claude |

Two lines of code swap the architecture.

In [ ]:
# ── Causal mask (decoder) ──────────────────────────────────────────────────────
causal_mask = np.triu(np.full((T, T), -np.inf), k=1)  # upper triangle = -inf
print("Causal mask:\n", causal_mask)

output_dec, weights_dec = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# ── Side-by-side comparison ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(weights_enc, ax=ax1, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names, vmin=0)
ax1.set_title('ENCODER — full attention\n(every feature sees every other)', fontweight='bold')
ax1.set_xlabel('Key'); ax1.set_ylabel('Query')

sns.heatmap(weights_dec, ax=ax2, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=feature_names, yticklabels=feature_names, vmin=0)
ax2.set_title('DECODER — causal mask\n(each feature sees only itself and earlier features)', fontweight='bold')
ax2.set_xlabel('Key'); ax2.set_ylabel('Query')

plt.suptitle('One mask → two architectures → BERT vs GPT', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Multi-Head Attention

One set of Q, K, V learns one relationship pattern. Run H heads in parallel, each with its own projections (dimension `d_k = d_model / H`), then concatenate and project back.

One head might track income-location correlations. Another might track occupancy-room-count interactions. Each head specialises independently.

In [ ]:
def multi_head_attention(X, num_heads, d_model):
    """Minimal multi-head attention — returns output and all head weight matrices."""
    d_k = d_model // num_heads
    rng = np.random.default_rng(0)

    all_heads = []
    all_weights = []
    for h in range(num_heads):
        WQ_h = rng.normal(0, 0.1, (d_model, d_k))
        WK_h = rng.normal(0, 0.1, (d_model, d_k))
        WV_h = rng.normal(0, 0.1, (d_model, d_k))
        Q_h = X @ WQ_h
        K_h = X @ WK_h
        V_h = X @ WV_h
        out_h, w_h = scaled_dot_product_attention(Q_h, K_h, V_h)
        all_heads.append(out_h)
        all_weights.append(w_h)

    # Concat heads and project back
    concat = np.concatenate(all_heads, axis=-1)  # (T, d_model)
    WO = rng.normal(0, 0.1, (d_model, d_model))
    output = concat @ WO                          # (T, d_model)
    return output, all_weights

MH_OUTPUT, head_weights = multi_head_attention(x_demo, num_heads=4, d_model=D_MODEL)
print(f"Multi-head output shape: {MH_OUTPUT.shape}")

# Plot all 4 heads
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for h, (ax, w) in enumerate(zip(axes, head_weights)):
    sns.heatmap(w, ax=ax, cmap='Blues', annot=True, fmt='.2f',
                xticklabels=feature_names, yticklabels=feature_names, vmin=0)
    ax.set_title(f'Head {h+1}', fontweight='bold')
    ax.set_xlabel('Key'); ax.set_ylabel('Query')

plt.suptitle('Multi-Head Attention — 4 heads, each learning different feature relationships', y=1.02)
plt.tight_layout()
plt.show()

## Full Transformer Encoder in Keras

Now we wire together the full encoder: token projection → positional encoding → N encoder blocks (LayerNorm + Multi-Head Attention + residual + FFN + residual) → global average pool → regression head.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

def build_tabular_transformer(T=8, d_in=1, d_model=32, num_heads=4,
                               num_layers=2, ffn_dim=64, dropout=0.1):
    inputs = keras.Input(shape=(T, d_in), name='feature_tokens')

    # Project each 1-dim token to d_model
    x = layers.Dense(d_model, name='token_projection')(inputs)

    # Add positional encoding (pre-computed, not learned)
    pe = positional_encoding(T, d_model).astype('float32')
    x = x + pe[np.newaxis, :, :]   # broadcast over batch dimension

    # Encoder blocks
    for i in range(num_layers):
        # Pre-LN Multi-Head Attention
        z = layers.LayerNormalization(epsilon=1e-6, name=f'ln_attn_{i}')(x)
        z = layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=d_model // num_heads,
                dropout=dropout,
                name=f'mha_{i}')(z, z)
        x = layers.Add(name=f'res_attn_{i}')([x, z])

        # Pre-LN Feed-Forward
        z = layers.LayerNormalization(epsilon=1e-6, name=f'ln_ffn_{i}')(x)
        z = layers.Dense(ffn_dim, activation='relu', name=f'ffn1_{i}')(z)
        z = layers.Dropout(dropout, name=f'ffn_drop_{i}')(z)
        z = layers.Dense(d_model, name=f'ffn2_{i}')(z)
        x = layers.Add(name=f'res_ffn_{i}')([x, z])

    # Pool across T tokens and regress
    x = layers.GlobalAveragePooling1D(name='mean_pool')(x)
    x = layers.Dense(32, activation='relu', name='head_dense')(x)
    outputs = layers.Dense(1, name='regression_output')(x)

    return keras.Model(inputs, outputs, name='TabularTransformer')

model = build_tabular_transformer()
model.summary()

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tokens, y.astype(np.float32), test_size=0.2, random_state=42)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr,
    epochs=100, batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0)

print(f"Stopped at epoch {len(history.history['loss'])}")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train loss (MSE)')
ax1.plot(history.history['val_loss'], label='Val loss (MSE)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE'); ax1.set_title('Loss curves')
ax1.legend()

ax2.plot(history.history['rmse'], label='Train RMSE')
ax2.plot(history.history['val_rmse'], label='Val RMSE')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('RMSE ($100k)'); ax2.set_title('RMSE curves')
ax2.legend()

plt.tight_layout(); plt.show()

# Final evaluation
results = model.evaluate(X_te, y_te, verbose=0)
print(f"\nTest MSE:  {results[0]:.4f}")
print(f"Test RMSE: {results[1]:.4f}  (in $100k units — so ${results[1]*100:.0f}k typical error)")

## Parameter Count: LSTM vs Transformer

In [ ]:
# Build equivalent LSTM for comparison
lstm_model = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.LSTM(32),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
], name='LSTM_baseline')
lstm_model.compile(optimizer='adam', loss='mse',
                   metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])

tr_params = model.count_params()
lstm_params = lstm_model.count_params()

print("=" * 50)
print(f"  Transformer params : {tr_params:>8,}")
print(f"  LSTM params        : {lstm_params:>8,}")
print("=" * 50)
print()
print("Key difference — not parameters, but computation structure:")
print("  LSTM : 8 sequential steps  → cannot parallelise across tokens")
print("  Transformer : 1 parallel pass → full GPU utilisation at any sequence length")

## The Hyperparameter Dial

In [ ]:
# Sweep d_model — the single most impactful dial for small transformers
results_sweep = []
for d in [8, 16, 32, 64]:
    m = build_tabular_transformer(d_model=d, num_heads=4 if d >= 16 else 2, ffn_dim=d*4)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse',
              metrics=[keras.metrics.RootMeanSquaredError(name='rmse')])
    h = m.fit(X_tr, y_tr, epochs=40, batch_size=256,
              validation_split=0.15, verbose=0)
    val_rmse = min(h.history['val_rmse'])
    params = m.count_params()
    results_sweep.append({'d_model': d, 'params': params, 'val_rmse': val_rmse})
    print(f"d_model={d:3d}  params={params:6,}  best val RMSE={val_rmse:.4f}")

# Plot
ds     = [r['d_model']  for r in results_sweep]
rmses  = [r['val_rmse'] for r in results_sweep]
params = [r['params']   for r in results_sweep]

fig, ax1 = plt.subplots(figsize=(8, 4))
color1 = 'steelblue'
ax1.plot(ds, rmses, 'o-', color=color1, label='Val RMSE')
ax1.set_xlabel('d_model'); ax1.set_ylabel('Val RMSE ($100k)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'coral'
ax2.plot(ds, params, 's--', color=color2, label='# Parameters')
ax2.set_ylabel('Parameter count', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('d_model sweep — accuracy vs parameter cost')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout(); plt.show()

## Exercises

1. **Change the mask.** In the encoder/decoder side-by-side cell, modify the causal mask to instead mask out only the **diagonal** (a token cannot attend to itself). Re-run the heatmap. Does the model still make sense? When would this be useful?

2. **Increase heads.** Change `num_heads` from 4 to 8 in `build_tabular_transformer`. What happens to `key_dim`? What constraint does this impose on `d_model`? Try `d_model=24, num_heads=8` — does it error? Why?

3. **Remove positional encoding.** Comment out the `x = x + pe[...]` line. Retrain and compare val RMSE. Does it matter for tabular data? Why would it matter more for natural language?

## Bridge

Ch.10 built the transformer encoder from first principles — attention, positional encoding, residuals, LayerNorm, and the one-mask difference between BERT and GPT. This is the architecture that every embedding model, every LLM, and every foundation model runs on.

The AI track's `RAGAndEmbeddings` note picks up exactly here: embedding models *are* transformer encoders trained with contrastive loss to produce sentence vectors you can compare by cosine similarity. The pooling step in those notes (mean-pool across token outputs) is exactly the `GlobalAveragePooling1D` layer at the end of this chapter's model.

> *One architecture. Three deployment patterns: encode for embeddings (BERT), generate autoregressively (GPT), or condition on a separate encoder (T5-style encoder-decoder).*